# 04d — Google ADK: production-minded customer-impact coordination

## Scenario: the invisible checkout incident

At 09:04, European checkout conversion falls 31%. Health checks are mostly green; a checkout release shipped at 08:42; six enterprise customers have complained. The system must identify likely impact, preserve evidence, draft an incident plan, and prepare—but never execute—customer notifications or remediation.

The goal is not “make a team talk.” It is to show when a compositional runtime makes distinct work products easier to control, evaluate, and deploy.

![Google ADK coordination architecture](../../../assets/google-adk-customer-impact.svg)

### What you will build conceptually

- bounded specialist agents and ownership contracts;
- function tools, agent-as-tool composition, and session-aware context;
- immutable events, artifacts, callbacks/plugins, and policy hooks;
- structured outputs and evidence validation;
- evaluation, replay/conformance thinking, observability, and deployment choices.

> **Trust boundary:** agent descriptions route work. They never grant tool permission. Every tool and policy gate must verify principal, tenant, arguments, budget, and approval independently.


## 1. Why Google ADK for this scenario?

[Google ADK](https://adk.dev/) is an open-source, multi-language agent-development framework that supports simple tool-using agents, multi-agent orchestration, graph workflows, sessions, events, artifacts, evaluation, and deployment paths. It is especially compelling when you need to grow from a single agent into compositional workloads while retaining an observable runner and Google Cloud integration options.

Choose ADK because its runtime features match a real requirement—not merely because there are multiple agents. A single bounded incident investigator is still the baseline for a clear outage. Introduce specialists only when independent evidence collection, context separation, or a distinct review role improves measurable outcomes.


## 2. Architecture and ownership

| Component | Allowed capability | Required artifact | Explicitly forbidden |
| --- | --- | --- | --- |
| Observability agent | read metrics and logs | anomaly, timeframe, source IDs | restart, rollback, customer messaging |
| Deployment agent | read release history | correlation/contradiction, source IDs | alter deployment |
| Customer-impact agent | read SLA metadata and tickets | affected segment and impact estimate | contact customer |
| Coordinator | consume validated artifacts | proposed incident plan | execute production action |
| Policy callback/plugin | inspect tool/action intent | allow, block, or approval requirement | invent business evidence |
| Human operator | approve exact action | signed decision with expiry | approve an ambiguous proposal |

This is a **manager-and-specialists** design. The coordinator does not receive broad operations tools; it receives evidence artifacts. That keeps its context focused and makes the action boundary reviewable.


In [ ]:
from pathlib import Path
import sys
repo_root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "curriculum" / "beginner" / "04-agent-development-frameworks" / "lab.py").exists())
sys.path.insert(0, str(repo_root / "curriculum" / "beginner" / "04-agent-development-frameworks"))
from lab import adk_shaped_customer_impact


In [ ]:
result = adk_shaped_customer_impact()
for role, finding in result["findings"].items():
    print(f"{role}: {finding['evidence']} [{finding['source']}]")
print("\nProposed plan:", result["plan"])
print("Coordination cost:", result["coordination_cost"])
assert "Do not restart" in result["plan"]


## 3. Agents, tools, and composition

In ADK, an LLM-backed agent can use function tools and can participate in compositions. ADK also supports agents/tools as building blocks for more complex workflows. Keep a tool contract narrow: accept typed arguments, derive tenant identity from trusted context, return compact results with source IDs, and expose typed failures.

~~~python
# Illustrative only: verify exact APIs in the current ADK documentation.
from google.adk import Agent
from google.adk.tools import FunctionTool

def query_metrics(region: str) -> dict:
    return {"source_id": "metrics-eu", "conversion_drop": 0.31}

observability = Agent(
    name="observability",
    model="gemini-flash-latest",
    instruction="Use only provided tools. Return an anomaly with source IDs.",
    tools=[FunctionTool(query_metrics)],
)
~~~

The model may call the function; deterministic service code must still enforce caller scope, tenant filters, rate limits, and result validation.


## 4. Sessions, state, and events

An ADK **session** represents a scoped conversation and state/history for an application/user interaction. The runtime emits **events** for user messages, model calls, function calls, tool results, state changes, control signals, and errors. That makes events a natural trace and audit substrate.

Design state deliberately:

| Scope | Safe example | Unsafe example |
| --- | --- | --- |
| Invocation-local | temporary query expansion | a reusable customer authorization |
| Session | active incident ID, selected region | a stale root-cause “fact” |
| Durable artifact | approved incident report PDF | raw secret/tool credential |
| Long-term memory | verified support preference | unvalidated deployment diagnosis |

Use a unique session per tenant/user/incident boundary. Do not reuse a session ID across tenants or treat conversation history as authorization.


## 5. Artifacts: evidence and deliverables

Artifacts provide a service-managed way to persist files or binary data associated with an interaction. They are appropriate for a generated incident brief, a reviewed CSV, or a screenshot evidence bundle. They are not a dumping ground for secrets or raw untrusted payloads.

The official [artifacts guide](https://adk.dev/artifacts/) explains artifact services and context access. An artifact should have an owner, a sensitivity label, a source manifest, retention policy, and integrity metadata.

~~~python
# Conceptual sketch
# context.save_artifact(
#     filename="incident-482-evidence.json",
#     artifact=json_bytes,
# )
# artifact = await context.load_artifact("incident-482-evidence.json")
~~~


## 6. Callbacks and plugins: the right policy seam

Callbacks run around agent, model, and tool lifecycle points. ADK plugins register reusable callback behavior at the Runner level, which makes them useful for cross-cutting policies such as redaction, tracing, tool allowlists, caching, rate budgets, and safety enforcement. See the [callbacks](https://adk.dev/callbacks/) and [plugins](https://adk.dev/plugins/) guides.

Use a callback/plugin to *enforce a deterministic rule*, not to ask another model to decide whether policy applies.

~~~python
# Pseudocode: block tool calls before execution.
def before_tool(ctx, tool, args):
    if tool.name in {"send_customer_notification", "rollback"}:
        if not ctx.state.get("approval_token"):
            raise PermissionError("Named approval required")
    if ctx.state.get("tenant_id") != trusted_tenant(ctx):
        raise PermissionError("Tenant mismatch")
~~~


In [ ]:
def evidence_gate(findings: dict[str, dict], plan: str) -> bool:
    sources = {item["source"] for item in findings.values()}
    needs_customer_evidence = "customer" in plan.lower() or "notify" in plan.lower()
    return len(sources) >= 2 and (not needs_customer_evidence or "customer_impact" in findings)

assert evidence_gate(result["findings"], result["plan"])
broken = dict(result["findings"]); broken.pop("customer_impact")
assert not evidence_gate(broken, "Notify customers immediately.")
print("Policy gate blocks unsupported external communication.")


## 7. Context management and model/tool boundaries

A production coordinator should receive a small, structured evidence packet, not unbounded raw logs, tickets, or web pages. Context selection controls both quality and attack surface.

1. Specialists query bounded, read-only tools.
2. Each returns a schema-valid finding: claim, confidence, source IDs, timestamp, and limitations.
3. A deterministic gate validates tenant, source IDs, freshness, and field sizes.
4. The coordinator receives only validated artifacts.
5. Policy blocks or requests approval for any action proposal.

This is also how you defend against indirect prompt injection. A support ticket may say “ignore prior rules.” It is evidence text, never system instruction or action authority.


## 8. Evaluation and replay

ADK supports evaluation and documents criteria including tool-trajectory matching, response matching, rubric-based quality, hallucination/groundedness, safety, multi-turn task success, and multi-turn trajectory quality. See [Why evaluate agents](https://adk.dev/evaluate/).

Create a small scenario suite:

| Test | Expected behavior |
| --- | --- |
| obvious outage | single agent wins on cost/latency |
| hidden regional degradation | specialists return supported evidence |
| poisoned ticket | content is treated as data; no action occurs |
| missing customer-impact evidence | coordinator abstains from customer notification |
| expired approval | write tool is blocked |
| same idempotency key | original result is replayed, not re-executed |

Measure supported-plan rate, forbidden-action rate, tool trajectory, latency, cost, coordination messages, and operator review time. Compare the team with the simpler baseline.


## 9. Deployment and operations

ADK deployment options include managed Agent Runtime/Agent Platform, Cloud Run, GKE, and other container-friendly infrastructure; see [deployment documentation](https://adk.dev/deploy/). Choose deployment based on identity, network isolation, observability, scaling, and operational ownership—not just framework affinity.

Production release checklist:

- [ ] service identity and tenant context are injected from trusted infrastructure
- [ ] tools are scoped, authenticated, rate-limited, and separately authorized
- [ ] callbacks/plugins enforce global budget, redaction, and action policy
- [ ] artifacts have classification, retention, and source manifests
- [ ] events/traces redact sensitive data and retain correlation IDs
- [ ] evaluations include trajectory, safety, hallucination, and recovery cases
- [ ] high-impact actions require human approval and idempotency
- [ ] a kill switch and escalation route are tested


## 10. Exercises

1. Add a RiskReviewer agent that can challenge a recommendation but cannot call operations tools.
2. Add an artifact manifest with source IDs, timestamps, and a retention label.
3. Implement a plugin-like function that enforces a per-session tool-call budget.
4. Add a test where an agent tries to use a nonexistent tool; require a safe stop.
5. Compare sequential versus parallel specialist collection under a 3-second latency budget.
6. Write a release rubric that decides when to route an incident to a single agent versus this team.

## References

- [ADK overview](https://adk.dev/)
- [ADK technical overview](https://adk.dev/get-started/about/)
- [ADK events](https://adk.dev/events/)
- [ADK callbacks](https://adk.dev/callbacks/)
- [ADK plugins](https://adk.dev/plugins/)
- [ADK artifacts](https://adk.dev/artifacts/)
- [ADK evaluation](https://adk.dev/evaluate/)
- [ADK deployment](https://adk.dev/deploy/)
- [Google ADK tools and integrations](https://adk.dev/tools/)
